# Etapa 3. Homogeneidade e tendência

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcoliveira-utfpr/spi_oni_pr/blob/main/notebooks/03_homogeneidade.ipynb)

O SPI compara a chuva de cada mês com a de todos os anos da série. Isso só faz sentido se a série for **homogênea**
(sem "degraus" causados por troca de pluviômetro ou mudança de local) e se o clima de 1981 for comparável ao de 2019.
Aqui aplicamos, na chuva **anual** de cada estação:

- o **teste de Pettitt**: existe um ano em que a série "muda de patamar"?
- o **teste de Mann-Kendall**: a chuva tende a aumentar ou a diminuir ao longo dos anos?
- a **inclinação de Sen**: de quantos mm por década é essa tendência?

Os três testes estão em `src/estatistica.py`, escritos com laços simples. Abra o arquivo no GitHub e leia: cada linha
está comentada. O capítulo 3 do caderno de aprendizagem faz o Mann-Kendall à mão.

In [ ]:
# ==========================================================================================
# CÉLULA 0. PREPARAR O AMBIENTE (rode sempre primeiro, toda vez que abrir o notebook)
# ==========================================================================================
# Esta célula:
#   1) define DE ONDE os dados serão lidos (direto do GitHub, no Colab);
#   2) baixa as funções do projeto (pasta src/ do GitHub): fdp.py, spi.py, enos.py, estatistica.py;
#   3) conecta o seu Google Drive e usa a pasta spi_oni_pr/resultados, onde ficam os resultados
#      de uma etapa para a outra.
import sys
import urllib.request
from pathlib import Path

NO_COLAB = "google.colab" in sys.modules
REPO = "https://raw.githubusercontent.com/fcoliveira-utfpr/spi_oni_pr/main/"

if Path("../dados").exists():                   # computador do orientador, dentro da pasta do projeto
    DADOS = "../dados/"
    sys.path.append("../src")
else:                                           # Colab
    DADOS = REPO + "dados/"
    Path("src").mkdir(exist_ok=True)
    for arquivo in ["fdp.py", "spi.py", "enos.py", "estatistica.py"]:
        urllib.request.urlretrieve(REPO + "src/" + arquivo, "src/" + arquivo)   # baixa o arquivo
    sys.path.append("src")

if NO_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    SAIDA = Path("/content/drive/MyDrive/spi_oni_pr/resultados")
else:
    SAIDA = Path("../resultados")
SAIDA.mkdir(parents=True, exist_ok=True)
print("Dados lidos de:", DADOS)
print("Resultados gravados em:", SAIDA)

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd

est = pd.read_csv(DADOS + "estacoes_pr.csv", dtype={"cod": str, "cod_ibge": str})
mensal = pd.read_csv(DADOS + "chuva_mensal_pr.csv", dtype={"cod": str})
meso = gpd.read_file(DADOS + "pr_mesorregioes.geojson")
destaque = est[est["destaque"].notna()]           # as 4 estações em destaque
print(len(est), "estações;", len(mensal), "linhas de chuva mensal")

# Aparência padrão dos gráficos do trabalho (rode uma vez; vale para todos os gráficos do notebook)
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.size": 10,
    "axes.spines.top": False,       # tira a borda de cima
    "axes.spines.right": False,     # tira a borda da direita
    "axes.grid": True,              # grade de fundo...
    "grid.color": "#e6e5e0",        # ...bem clara, para não competir com os dados
    "grid.linewidth": 0.6,
    "axes.axisbelow": True,         # grade atrás das barras e linhas
})

AZUL = "#2a78d6"                    # cor única quando há só uma série
# uma cor fixa para cada estação em destaque (a cor acompanha a estação em todas as figuras)
CORES = {"Santa Helena": "#2a78d6", "Cascavel": "#eb6834", "Foz do Iguaçu": "#1baf7a", "Mercedes": "#eda100"}

from matplotlib.colors import LinearSegmentedColormap

# escala divergente: vermelho (seco / negativo) - cinza (zero) - azul (úmido / positivo)
DIVERGENTE = LinearSegmentedColormap.from_list("div", ["#a3201f", "#e34948", "#f0efec", "#2a78d6", "#184f95"])
# cores das fases do ENOS: El Niño = águas quentes (vermelho), La Niña = águas frias (azul)
CORES_FASE = {"El Niño": "#e34948", "Neutro": "#9a9890", "La Niña": "#2a78d6"}

def mapa_base(ax):
    """Desenha os limites das mesorregiões no fundo de um mapa."""
    meso.plot(ax=ax, color="#f4f3ef", edgecolor="#9a9890", linewidth=0.6)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    for lado in ["left", "bottom"]:
        ax.spines[lado].set_visible(False)

import estatistica as E

## 1. Chuva anual de cada estação

Só os anos completos (1981 a 2019). O resultado é uma tabela com uma linha por estação e por ano.

In [ ]:
anual = mensal[mensal["ano"] <= 2019].groupby(["cod", "ano"])["chuva_mm"].sum().reset_index()
anual.head()

## 2. Os testes, estação por estação

O laço passa pelas 370 estações. Para cada uma, separa a série anual e aplica os três testes. Demora uns segundos.

In [ ]:
linhas = []
for cod in est["cod"]:
    serie = anual[anual["cod"] == cod].sort_values("ano")
    chuva = list(serie["chuva_mm"])
    anos = list(serie["ano"])

    posicao, k, p_pettitt = E.pettitt(chuva)          # posição = último ano ANTES da mudança
    s, z, p_mk = E.mann_kendall(chuva)
    inclinacao = E.sen(chuva) * 10                    # mm por ano -> mm por década

    linhas.append({"cod": cod,
                   "ano_mudanca": anos[posicao + 1],  # primeiro ano DEPOIS da mudança
                   "p_pettitt": p_pettitt,
                   "S_mk": s,
                   "p_mk": p_mk,
                   "sen_mm_decada": inclinacao})

testes = pd.DataFrame(linhas).merge(est[["cod", "municipio", "mesorregiao", "lat", "lon"]], on="cod")
testes["mudanca"] = testes["p_pettitt"] < 0.05                 # True = ponto de mudança significativo
testes["tendencia"] = "sem"
testes.loc[(testes["p_mk"] < 0.05) & (testes["S_mk"] > 0), "tendencia"] = "aumento"
testes.loc[(testes["p_mk"] < 0.05) & (testes["S_mk"] < 0), "tendencia"] = "redução"
testes.to_csv(SAIDA / "testes_homogeneidade.csv", index=False)

print("Estações com ponto de mudança (Pettitt, 5%):", testes["mudanca"].sum())
print(testes["tendencia"].value_counts())

## 3. Resumo por mesorregião (Tabela 3 do TCC)

In [ ]:
tabela3 = testes.groupby("mesorregiao").agg(
    estacoes=("cod", "count"),
    com_mudanca=("mudanca", "sum"),
    tendencia_aumento=("tendencia", lambda t: (t == "aumento").sum()),
    tendencia_reducao=("tendencia", lambda t: (t == "redução").sum()),
    sen_medio_mm_decada=("sen_mm_decada", "mean"),
).round(1)
tabela3["pct_mudanca"] = (tabela3["com_mudanca"] / tabela3["estacoes"] * 100).round(0)
tabela3.to_csv(SAIDA / "tabela03_homogeneidade_tendencia.csv")
tabela3

Em que anos aparecem os pontos de mudança? Se muitos caem no mesmo ano e na mesma região, pode ser um sinal do
clima; se estão espalhados, é mais provável que sejam problemas de cada estação.

In [ ]:
testes.loc[testes["mudanca"], "ano_mudanca"].value_counts().sort_index()

## 4. Mapas (Figura 3 do TCC)

Painel (a): estações com ponto de mudança significativo. Painel (b): inclinação de Sen das estações com tendência
significativa, em escala divergente (vermelho = redução, azul = aumento); as demais aparecem em cinza claro.

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(13, 5))

ax = eixos[0]
mapa_base(ax)
sem = testes[~testes["mudanca"]]
com = testes[testes["mudanca"]]
ax.scatter(sem["lon"], sem["lat"], s=8, color="#c3c2b7", label=f"Sem mudança ({len(sem)})")
ax.scatter(com["lon"], com["lat"], s=22, color="black", marker="^", label=f"Com mudança ({len(com)})")
ax.set_title("(a) Ponto de mudança (Pettitt, 5%)", fontsize=10)
ax.legend(loc="lower left", fontsize=8)

ax = eixos[1]
mapa_base(ax)
nao_sig = testes[testes["tendencia"] == "sem"]
sig = testes[testes["tendencia"] != "sem"]
ax.scatter(nao_sig["lon"], nao_sig["lat"], s=8, color="#d9d8d2", label=f"Sem tendência ({len(nao_sig)})")
limite = max(abs(testes["sen_mm_decada"].min()), abs(testes["sen_mm_decada"].max()))
pontos = ax.scatter(sig["lon"], sig["lat"], c=sig["sen_mm_decada"], cmap=DIVERGENTE, vmin=-limite, vmax=limite,
                    s=26, edgecolor="black", linewidth=0.3, label=f"Com tendência ({len(sig)})")
fig.colorbar(pontos, ax=ax, label="Inclinação de Sen (mm por década)", shrink=0.7)
ax.set_title("(b) Tendência da chuva anual (Mann-Kendall, 5%)", fontsize=10)
ax.legend(loc="lower left", fontsize=8)

fig.tight_layout()
fig.savefig(SAIDA / "fig03_homogeneidade_tendencia.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. As estações em destaque

A série anual de cada estação em destaque, com a reta de Sen (a inclinação passando pela mediana dos dados).

In [ ]:
fig, eixos = plt.subplots(2, 2, figsize=(12, 6), sharey=True)
for ax, (i, linha) in zip(eixos.flat, destaque.iterrows()):      # zip anda nas duas listas juntas
    nome = linha["municipio"]
    serie = anual[anual["cod"] == linha["cod"]].sort_values("ano")
    r = testes[testes["cod"] == linha["cod"]].iloc[0]
    ax.plot(serie["ano"], serie["chuva_mm"], color=CORES[nome], linewidth=2, marker="o", markersize=3)
    # reta de Sen: passa pelo ponto (ano mediano, chuva mediana) com a inclinação de Sen
    inclinacao_ano = r["sen_mm_decada"] / 10
    x = serie["ano"]
    y = serie["chuva_mm"].median() + inclinacao_ano * (x - x.median())
    ax.plot(x, y, color="black", linestyle="--", linewidth=1)
    ax.set_title(f"{nome}: Sen = {r['sen_mm_decada']:.0f} mm/década (p = {r['p_mk']:.2f})", fontsize=9)
for ax in eixos[:, 0]:
    ax.set_ylabel("Chuva anual (mm)")
fig.tight_layout()
plt.show()

---

## O que observar

- Uma estação com ponto de mudança **não é automaticamente ruim**: mudanças climáticas reais (por exemplo, uma
  década mais chuvosa) também aparecem no Pettitt. Olhe o mapa: pontos agrupados numa região sugerem clima;
  pontos isolados, problema da estação.
- Com 39 anos, uma tendência precisa ser forte para ser significativa. É normal que a maioria das estações não
  tenha tendência.

## Ponto de decisão com o orientador

O que fazer com as estações com ponto de mudança: excluir do estudo, ou manter e registrar? Leve a lista
(`testes_homogeneidade.csv`, no Drive) para a reunião.

## Exercícios

**Exercício 1.** Quantas estações das quatro em destaque têm ponto de mudança? E tendência? Escreva um parágrafo
para o item 5.2 do TCC com esses resultados.

In [ ]:
# escreva sua resposta aqui

**Exercício 2.** Escolha uma estação com ponto de mudança significativo e faça o gráfico da série anual dela,
marcando o ano da mudança com `ax.axvline(ano, color="black")`. A mudança parece real?

In [ ]:
# escreva sua resposta aqui

---

**Entregar.** `fig03`, `tabela03` e os exercícios. **O que escrever:** item 5.2 do TCC2.